In [0]:
%sql
USE CATALOG industry

# Clean the Raw Data

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW country_silver AS
SELECT
  country_name as country,
  country_name as etl_business_key,
  etl_filename,
  SHA2(country_name, 256) AS etl_record_hash
FROM
  bronze.country
ORDER BY
  country_name

# MERGE TO Silver Table

In [0]:
%sql
MERGE INTO
  silver.country as tgt
USING
  country_silver as src
ON
  tgt.etl_business_key = src.etl_business_key
  AND tgt.etl_record_hash <> src.etl_record_hash
WHEN MATCHED THEN UPDATE SET
  tgt.country = src.country,
  tgt.etl_record_hash = src.etl_record_hash,
  tgt.etl_update_timestamp = NOW()
WHEN NOT MATCHED THEN INSERT (
    tgt.country,
    tgt.etl_business_key,
    tgt.etl_record_hash,
    tgt.etl_filename
  )
  VALUES (
    src.country,
    src.etl_business_key,
    src.etl_record_hash,
    src.etl_filename
  )